In [ ]:
pip install gurobipy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.4/14.4 MB 86.0 MB/s eta 0:00:00


In [90]:
from gurobipy import Model, GRB, quicksum

# Define model
model = Model("Block_Allocation")

In [91]:
# Define sets
n = 10  # Number of blocks
w = 5  # Number of stacks
t = 5  # Number of periods
h = [3, 4, 2, 5, 1]  # Capacity of each stack

In [92]:
# Define variables
x = model.addVars(n, w, t - 1, vtype=GRB.BINARY, name="x")  # Block i in stack j at period k
y = model.addVars(n, w, t - 1, vtype=GRB.BINARY, name="y")  # Block i moved from stack j at period k

In [93]:
# Define objective function
obj = quicksum(y[i, j, k] for i in range(n) for j in range(w) for k in range(t - 1))
model.setObjective(obj, GRB.MINIMIZE)

In [94]:
# Define constraints
# Constraint 1: Each block must be allocated to one stack at each period
model.addConstrs(quicksum(x[i, j, k] for j in range(w)) == 1 for i in range(n) for k in range(t - 1))

# Constraint 2: Capacity of each stack
model.addConstrs(quicksum(x[i, j, k] for i in range(n)) <= h[j] for j in range(w) for k in range(t - 1))

# Constraint 3: Movement of blocks - activated if block was in stack j at period k and not in period k+1
model.addConstrs(y[i, j, k] >= x[i, j, k] - x[i, j, (k + 1) % (t-1)]  for i in range(n) for j in range(w) for k in range(t - 1))

# Constraint 4: Block cannot be moved if not in the stack at period k
model.addConstrs(y[i, j, k] <= x[i, j, k] for i in range(n) for j in range(w) for k in range(t - 1))

{(0, 0, 0): <gurobi.Constr *Awaiting Model Update*>,
 (0, 0, 1): <gurobi.Constr *Awaiting Model Update*>,
 (0, 0, 2): <gurobi.Constr *Awaiting Model Update*>,
 (0, 0, 3): <gurobi.Constr *Awaiting Model Update*>,
 (0, 1, 0): <gurobi.Constr *Awaiting Model Update*>,
 (0, 1, 1): <gurobi.Constr *Awaiting Model Update*>,
 (0, 1, 2): <gurobi.Constr *Awaiting Model Update*>,
 (0, 1, 3): <gurobi.Constr *Awaiting Model Update*>,
 (0, 2, 0): <gurobi.Constr *Awaiting Model Update*>,
 (0, 2, 1): <gurobi.Constr *Awaiting Model Update*>,
 (0, 2, 2): <gurobi.Constr *Awaiting Model Update*>,
 (0, 2, 3): <gurobi.Constr *Awaiting Model Update*>,
 (0, 3, 0): <gurobi.Constr *Awaiting Model Update*>,
 (0, 3, 1): <gurobi.Constr *Awaiting Model Update*>,
 (0, 3, 2): <gurobi.Constr *Awaiting Model Update*>,
 (0, 3, 3): <gurobi.Constr *Awaiting Model Update*>,
 (0, 4, 0): <gurobi.Constr *Awaiting Model Update*>,
 (0, 4, 1): <gurobi.Constr *Awaiting Model Update*>,
 (0, 4, 2): <gurobi.Constr *Awaiting Model Upd

In [95]:
# Solve the model
model.optimize()

# Print the solution
if model.status == GRB.OPTIMAL:
    print("Optimal solution found:")
    print("Objective value:", model.ObjVal)
    for i in range(n):
        for j in range(w):
            for k in range(t - 1):
                if x[i, j, k].x > 0.5:
                    print(f"Block {i} is in stack {j} at period {k}")
                if y[i, j, k].x > 0.5:
                    print(f"Block {i} is moved from stack {j} at period {k}")
else:
    print("No optimal solution found.")

Gurobi Optimizer version 12.0.0 build v12.0.0rc1 (linux64 - "Ubuntu 22.04.3 LTS")

CPU model: Intel(R) Xeon(R) CPU @ 2.20GHz, instruction set [SSE2|AVX|AVX2]
Thread count: 1 physical cores, 2 logical processors, using up to 2 threads

Optimize a model with 460 rows, 400 columns and 1400 nonzeros
Model fingerprint: 0x05e90b33
Variable types: 0 continuous, 400 integer (400 binary)
Coefficient statistics:
  Matrix range     [1e+00, 1e+00]
  Objective range  [1e+00, 1e+00]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 5e+00]
Found heuristic solution: objective 27.0000000
Presolve removed 200 rows and 0 columns
Presolve time: 0.01s
Presolved: 260 rows, 400 columns, 1000 nonzeros
Variable types: 0 continuous, 400 integer (400 binary)

Root relaxation: objective 0.000000e+00, 123 iterations, 0.00 seconds (0.00 work units)

    Nodes    |    Current Node    |     Objective Bounds      |     Work
 Expl Unexpl |  Obj  Depth IntInf | Incumbent    BestBd   Gap | It/Node Time

*    0